# Workflow: a simple linear workflow

This is notebook 1 of 3 in the workflow series.

A workflow is a lightweight container that lists process steps and records
their order. This notebook walks through the full pipeline for a simple
three-step quality assurance workflow:

```
Step 1: Raw Material Inspection
           |
Step 2: Tensile Test
           |
Step 3: Quality Report
```

After working through this notebook, see:
- [Notebook 2](2_workflow_branching.ipynb): branching and parallel steps
- [Notebook 3](3_workflow_cross_domain.ipynb): a full cross-domain example


In [1]:
%pip install -q semantic-schemas

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, json, pathlib, rdflib
from semantic_schemas import Schema

HERE     = pathlib.Path().resolve()   # schemas/workflow/PMDCo/docs/
WORKFLOW = HERE.parent                # schemas/workflow/PMDCo/

schema = Schema(WORKFLOW)

print("Python :", sys.executable)
print("Schema :", WORKFLOW)

Python : /root/semantic-dataspace/.venv/bin/python3
Schema : /root/semantic-dataspace/semantic-schemas/schemas/workflow/PMDCo


## Step 1: Define the workflow

Fill in a workflow name and a list of steps. Each step needs at minimum a label.
Optional fields:

| Field | Purpose |
|---|---|
| `step_type` | ontology class CURIE for the step (e.g. `obi:0000070` for an assay) |
| `description` | free-text description |
| `conditions` | inline parameters (name, value, unit) |
| `instance_iri` | link to a detailed record in a knowledge graph (covered in notebook 3) |
| `preceded_by` | explicit predecessor step IDs for branching (covered in notebook 2) |


In [3]:
workflow_input = {
    'workflow_name': 'QA workflow, batch A',
    'description': 'Three-step quality assurance workflow for a steel sample batch.',
    'steps': [
        {
            'label': 'Raw Material Inspection',
            'description': 'Visual and dimensional check of the incoming steel sheet.',
        },
        {
            'label': 'Tensile Test',
            'description': 'Room-temperature tensile test following ISO 6892-1.',
            'conditions': [
                {'name': 'Standard',    'unit': 'ISO 6892-1'},
                {'name': 'Temperature', 'value': 23, 'unit': 'degC'},
            ],
        },
        {
            'label': 'Quality Report',
            'description': 'Document test results and issue an accept or reject decision.',
        },
    ],
}

oold_doc = schema.transform(workflow_input)

print('OO-LD document:')
print(json.dumps(oold_doc, indent=2))

OO-LD document:
{
  "conforms_to": "https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/workflow/PMDCo/#v1.0.0",
  "type": "bfo:0000015",
  "id": "workflow-qa-workflow-batch-a",
  "label": "QA workflow, batch A",
  "has_part": [
    {
      "type": "bfo:0000015",
      "id": "workflow-qa-workflow-batch-a-step-1-raw-material-inspection",
      "label": "Raw Material Inspection",
      "description": "Visual and dimensional check of the incoming steel sheet."
    },
    {
      "type": "bfo:0000015",
      "id": "workflow-qa-workflow-batch-a-step-2-tensile-test",
      "label": "Tensile Test",
      "description": "Room-temperature tensile test following ISO 6892-1.",
      "preceded_by": [
        "workflow-qa-workflow-batch-a-step-1-raw-material-inspection"
      ],
      "has_process_condition": [
        {
          "type": "pmdco:PMD_0000013",
          "id": "workflow-qa-workflow-batch-a-step-2-tensile-test-param-1",
          "parameter_label": "Standard",
   

The transform automatically links each step to the one before it
(`preceded_by` pointing to the previous step ID). The first step has no predecessor.

Notice that step IDs are derived from the workflow name and step labels.
You can override them with `step_id` (shown in notebook 2).


In [4]:
print('Step order:')
for s in oold_doc['has_part']:
    pre = s.get('preceded_by', ['(first)'])
    print(f"  {s['id']}")
    print(f"    label       : {s['label']}")
    print(f"    preceded_by : {pre}")
    print()

Step order:
  workflow-qa-workflow-batch-a-step-1-raw-material-inspection
    label       : Raw Material Inspection
    preceded_by : ['(first)']

  workflow-qa-workflow-batch-a-step-2-tensile-test
    label       : Tensile Test
    preceded_by : ['workflow-qa-workflow-batch-a-step-1-raw-material-inspection']

  workflow-qa-workflow-batch-a-step-3-quality-report
    label       : Quality Report
    preceded_by : ['workflow-qa-workflow-batch-a-step-2-tensile-test']



The OO-LD document is parsed into an RDF graph using the ontology context
from `specs/schema.oold.yaml`. This maps JSON field names to ontology IRIs:
`has_part` becomes `bfo:BFO_0000051` and `preceded_by` becomes `bfo:BFO_0000062`.

In [5]:
flat = schema.to_graph(workflow_input)

print(f"Graph contains {len(flat)} triples.\n")
print(flat.serialize(format="turtle"))

Graph contains 27 triples.

@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix ns1: <http://qudt.org/schema/qudt/> .
@prefix ns2: <http://purl.obolibrary.org/obo/> .
@prefix ns3: <https://w3id.org/pmd/co/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://w3id.org/pmd/co/test/workflow-qa-workflow-batch-a> a ns2:BFO_0000015 ;
    rdfs:label "QA workflow, batch A" ;
    ns2:BFO_0000051 <https://w3id.org/pmd/co/test/workflow-qa-workflow-batch-a-step-1-raw-material-inspection>,
        <https://w3id.org/pmd/co/test/workflow-qa-workflow-batch-a-step-2-tensile-test>,
        <https://w3id.org/pmd/co/test/workflow-qa-workflow-batch-a-step-3-quality-report> ;
    dcterms:conformsTo <https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/workflow/PMDCo/#v1.0.0> ;
    rdfs:comment "Three-step quality assurance workflow for a steel sample batch." .

<https://w3id.org/pmd/co/test/workflow-qa-workflow-batch-a

## Step 3: Validate with SHACL

The SHACL shape checks that:
- The workflow node has a label and at least one step
- Each step node has a label

In [6]:
conforms, violations = schema.validate(flat)

print(f"Conforms: {conforms}")
for v in violations:
    print(f"  Violation: {v}")

Conforms: True


## Step 4: Query the graph

SPARQL is the query language for RDF graphs. The query below retrieves
all steps and their predecessors, showing the workflow order.


In [7]:
SPARQL = """
PREFIX bfo:    <http://purl.obolibrary.org/obo/BFO_>
PREFIX rdfs:   <http://www.w3.org/2000/01/rdf-schema#>
PREFIX dcterms: <http://purl.org/dc/terms/>

SELECT ?step_label ?preceded_by_label
WHERE {
  ?step rdfs:label ?step_label .
  FILTER NOT EXISTS { ?step dcterms:conformsTo ?_ . }  # exclude the workflow root
  OPTIONAL {
    ?step bfo:0000062 ?prev .
    ?prev rdfs:label ?preceded_by_label .
  }
}
ORDER BY ?step_label
"""

rows = list(flat.query(SPARQL))
print('{:<30}  {}'.format('Step', 'Preceded by'))
print('-' * 55)
for r in rows:
    pre = str(r.preceded_by_label) if r.preceded_by_label else '(first)'
    print(f'{str(r.step_label):<30}  {pre}')

Step                            Preceded by
-------------------------------------------------------
Quality Report                  Tensile Test
Raw Material Inspection         (first)
Standard                        (first)
Temperature                     (first)
Tensile Test                    Raw Material Inspection


## Summary

| Step | What happened |
|---|---|
| 1 | Defined a workflow with three steps as a plain Python dict |
| 2 | JSONata transform converted it to an OO-LD document |
| 3 | The OO-LD document was parsed into an RDF graph |
| 4 | SHACL validation confirmed the graph is structurally correct |
| 5 | A SPARQL query retrieved the step order from the graph |

---

**Next:** [Notebook 2](2_workflow_branching.ipynb) shows how to use `preceded_by`
to create branching and parallel workflows.
